<a href="https://colab.research.google.com/github/harship123-source/Pfizer-Advanced-AI-Powered-Document-Insights-Data-Extraction-Externship/blob/main/RAG_with_Open_Source_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries with CUDA support
!pip install -q torch

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


In [ ]:
# Check CUDA version first
!nvcc --version

# Install llama-cpp-python with CUDA 12.x support
!pip install --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu123
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 MB 231.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 168.0 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.2.90-cp313-cp313-linux_x86_64.whl size=3482658 sha256=2187a5739bb1b8769ef0c10b1cd48ea5f7eb6485b0a08d9adf21608f0720c377
  Stored in directory: /tmp/pip-ephem-wheel-cache-sf5u63x1/wheels/b0/5c/dc/f326b9846d6aac94e12ade7479700c96388f3cca5a2f351963
Successfully built llama-cpp-python


In [ ]:
!pip install llama-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 84.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.7/168.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 19.3 MB/s eta 0:00:00
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1


In [ ]:
from llama_cpp import Llama
import os

# Download Mistral model if not already present
model_path = "/content/mistral.gguf"
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Load the model with GPU acceleration
try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=20,  # Start with 1 layer on GPU to be safe
        n_ctx=2048,      # Context window size
        verbose=True     # Show loading progress
    )

    print("Model loaded successfully!")



except Exception as e:
    print(f"Error loading model: {e}")

--2026-09-17 12:10:30--  https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 99.86.101.64, 99.86.101.39, 99.86.101.36, ...
Connecting to huggingface.co (huggingface.co)|99.86.101.64|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65778ac662d3ac1817cc9201/865f5e4682dddb29c2e20270b2471a7590c83a414bbf1d72cf4c08fdff2eeca4?user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27mistral-7b-instruct-v0.2.Q4_K_M.gguf%3B+filename%3D%22mistral-7b-instruct-v0.2.Q4_K_M.gguf%22%3B&X-Xet-Cas-Uid=public&Expires=1789650630&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU3NzhhYzY2MmQzYWMxODE3Y2M5MjAxLzg2NWY1ZTQ2ODJkZGRiMjljMmUyMDI3MGIyNDcxYTc1OTBjODNhNDE0YmJmMWQ3MmNmNGMwOGZkZmYyZWVjYTRcXD91c2VyX2lkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomWC1YZXQtQ2Fz

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.

Model downloaded to /content/mistral.gguf
Model file exists. Size: 4166.07 MB


llama_model_loader: - kv  14:                      tokenizer.ggml.scores arr[f32,32000]   = [0.000000, 0.000000, 0.000000, 0.0000...
llama_model_loader: - kv  15:                  tokenizer.ggml.token_type arr[i32,32000]   = [2, 3, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, ...
llama_model_loader: - kv  16:                tokenizer.ggml.bos_token_id u32              = 1
llama_model_loader: - kv  17:                tokenizer.ggml.eos_token_id u32              = 2
llama_model_loader: - kv  18:            tokenizer.ggml.unknown_token_id u32              = 0
llama_model_loader: - kv  19:            tokenizer.ggml.padding_token_id u32              = 0
llama_model_loader: - kv  20:               tokenizer.ggml.add_bos_token bool             = true
llama_model_loader: - kv  21:               tokenizer.ggml.add_eos_token bool             = false
llama_model_loader: - kv  22:                    tokenizer.chat_template str              = {{ bos_token }}{% for message in mess...
llama_model_loader: - kv  23: 

Model loaded successfully!


Guessed chat format: mistral-instruct


In [ ]:
# Test with a RAG query
prompt = "What is RAG in the context of large language models?"
print(f"\nSending prompt: {prompt}")

response = llm(prompt, max_tokens=256, temperature=0.1)
print("\nResponse:")
print(response["choices"][0]["text"])


Sending prompt: What is RAG in the context of large language models?



llama_print_timings:        load time =    5899.95 ms
llama_print_timings:      sample time =       9.82 ms /   187 runs   (    0.05 ms per token, 19042.77 tokens per second)
llama_print_timings: prompt eval time =    5899.27 ms /    13 tokens (  453.79 ms per token,     2.20 tokens per second)
llama_print_timings:        eval time =  121405.89 ms /   186 runs   (  652.72 ms per token,     1.53 tokens per second)
llama_print_timings:       total time =  127481.04 ms /   199 tokens



Response:


RAG, or Recall and Agreement, is a metric used to evaluate the performance of large language models in answering factual questions. It measures the model's ability to recall the correct answer from a set of candidate answers and agree with human annotators on the correct answer.

The RAG metric is calculated as the average of recall and agreement scores. Recall is the percentage of questions for which the model provides the correct answer among all candidate answers. Agreement is the percentage of questions for which the model's answer matches the answer given by human annotators.

RAG is an important metric for evaluating the accuracy and reliability of large language models, as it provides a more nuanced assessment of their performance than simple accuracy metrics like precision and recall. It helps to identify situations where the model may provide plausible but incorrect answers, and highlights areas where the model needs improvement.


## Integrating Open-Source LLMs into RAG (with a PDF!)

In [ ]:
!pip install pymupdf
!pip install llama-index-llms-llama-cpp --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 78.4 MB/s eta 0:00:00


In [ ]:
!pip install llama-index-embeddings-huggingface

In [ ]:
import fitz  # PyMuPDF

# Load the sample contract PDF
pdf_path = "/content/sample-sdf-document.pdf"
doc = fitz.open(pdf_path)

# Extract text from all pages
text = "\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the PDF.")

Extracted 617 words from the PDF.


In [ ]:
from llama_index.core import VectorStoreIndex, Document, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.settings import Settings
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter


# Configure the LLM
llm = LlamaCPP(
    model_path="/content/mistral.gguf",
    temperature=0.3,  # Reduced temperature for less creative, more direct answers
    max_new_tokens=500, # Reduced to limit the length of the response
    context_window=2048,
    model_kwargs={"n_gpu_layers": -1}
)

# Configure open-source embedding model
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"  # Lightweight but effective embedding model
)

# Set as the default LLM and embedding model
Settings.llm = llm
Settings.embed_model = embed_model

# Create documents from your text

documents = [Document(text=text)]  # 'text' should be your document content

# Convert into nodes (smaller chunks)
text_splitter = SentenceSplitter(chunk_size=100, chunk_overlap=100)
nodes = text_splitter.get_nodes_from_documents(documents)

print(f"Created {len(nodes)} chunks from the document.")

# Build index
index = VectorStoreIndex.from_documents(nodes)

# Configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=2,  # Retrieve 2 most similar chunks
)

# Configure response synthesizer
response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
)


llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Created 21 chunks from the document.


In [ ]:
query = "What sterilization method was used for this product?"

# Assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

# Test query
response = query_engine.query(query)
print(response)


llama_print_timings:        load time =  117407.65 ms
llama_print_timings:      sample time =       3.59 ms /    71 runs   (    0.05 ms per token, 19777.16 tokens per second)
llama_print_timings: prompt eval time =  117407.17 ms /   291 tokens (  403.46 ms per token,     2.48 tokens per second)
llama_print_timings:        eval time =   45933.79 ms /    70 runs   (  656.20 ms per token,     1.52 tokens per second)
llama_print_timings:       total time =  163397.24 ms /   361 tokens


 The product was autoclaved at 121 degrees Celsius for more than 15 minutes and also subjected to gamma irradiation with a dose between 25.0 and 40.0 kGy. Both methods are commonly used for sterilization of medical devices and pharmaceutical products.
